# 01 — LLM Fundamentals

**AI Engineer — hands-on session**

This whole notebook uses the `openai` library against [OpenRouter](https://openrouter.ai/), with free models (`:free`).

The idea that runs through the whole session:

> **An LLM is a stateless function: text in, text out.**
> Everything else — memory, tools, RAG — is scaffolding we build around it.

---

### Before you start

1. Create an API key at https://openrouter.ai/keys
2. In Colab: **🔑 Secrets** panel (key icon) → new secret `OPENROUTER_API_KEY` → enable *Notebook access*
3. Run the cells in order

In [ ]:
%pip install -q openai gradio

In [ ]:
import os

try:
    from google.colab import userdata  # type: ignore

    API_KEY = userdata.get("OPENROUTER_API_KEY")
except ImportError:
    # Outside Colab: environment variable or .env
    API_KEY = os.environ["OPENROUTER_API_KEY"]

assert API_KEY, "Missing OPENROUTER_API_KEY"

In [ ]:
from openai import OpenAI

# In part 2 of the talk, this single line will point at LiteLLM on the NAS.
BASE_URL = "https://openrouter.ai/api/v1"
MODEL = "meta-llama/llama-3.3-70b-instruct:free"

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

---
## 1. The simplest possible call

A list of messages goes in, one message comes out. That is the entire API.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user", "content": "In one sentence, what is the capital of New Zealand?"},
    ],
)

print(response.choices[0].message.content)

### What it actually returns

The response is more than text. Look at `usage`: **that is what you pay for**.

In [ ]:
print("Model         :", response.model)
print("Finish reason :", response.choices[0].finish_reason)
print("Input tokens  :", response.usage.prompt_tokens)
print("Output tokens :", response.usage.completion_tokens)
print("Total tokens  :", response.usage.total_tokens)

---
## 2. Streaming

The model generates **one token at a time**. Streaming does not make it faster: it just lets us watch it happen.

In [ ]:
stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Explain what a token is, in 3 short bullet points."}],
    stream=True,
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="")

---
## 3. Temperature: it is a probability engine

At every step the model computes a probability for *every* possible next token. **Temperature** decides how much we respect that distribution:

- `temperature=0` → always the most likely token. Repeatable.
- High `temperature` → flattens the distribution, less likely options get a chance. Creative, or incoherent.

Same question, four times:

In [ ]:
PROMPT = "Invent a name for a coffee shop in Wellington. Reply with the name only."

for temperature in (0.0, 0.0, 1.5, 1.5):
    out = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": PROMPT}],
        temperature=temperature,
        max_tokens=20,
    )
    print(f"temperature={temperature} -> {out.choices[0].message.content.strip()}")

> The first two should match; the last two should not.
> There is no absolute guarantee: GPU batching introduces some non-determinism even at temperature 0.

---
## 4. The LLM has no memory

We give it a personal detail:

In [ ]:
first = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "My name is Miguel and I'm from Colombia."}],
)

print(first.choices[0].message.content)

### ❓ Before running: what do you think it will answer?

In [ ]:
second = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is my name?"}],
)

print(second.choices[0].message.content)

It does not remember. **This is not a bug: it is the design.** Every call is independent, like running a pure function.

The fix? There is no trick: we send everything again.

In [ ]:
conversation = [
    {"role": "user", "content": "My name is Miguel and I'm from Colombia."},
    {"role": "assistant", "content": first.choices[0].message.content},
    {"role": "user", "content": "What is my name?"},
]

third = client.chat.completions.create(model=MODEL, messages=conversation)

print(third.choices[0].message.content)

> ### 💡 The key idea
> **"Chat" does not exist.** It is a Python list we append to and resend in full on every turn.
>
> What ChatGPT calls *memory* is this loop: store the history and resend it.

Remember this when we get to RAG.

---
## 5. The cost of remembering

If we resend the whole conversation on every turn, on turn *n* we send *n* messages. The total does not grow in a straight line: it grows **quadratically** (≈ n²/2).

That is why long conversations get slow and expensive, and why a *context window* exists.

In [ ]:
turns = [
    "Give me one fun fact about New Zealand.",
    "Another one, please.",
    "One more, different topic.",
    "Another one.",
    "Last one.",
]

messages = [{"role": "system", "content": "Reply in exactly one short sentence."}]
sent_per_turn, cumulative = [], []
total = 0

for turn in turns:
    messages.append({"role": "user", "content": turn})
    result = client.chat.completions.create(model=MODEL, messages=messages)
    messages.append({"role": "assistant", "content": result.choices[0].message.content})

    total += result.usage.total_tokens
    sent_per_turn.append(result.usage.prompt_tokens)
    cumulative.append(total)
    print(f"Turn {len(cumulative)}: sent {result.usage.prompt_tokens:>4} | cumulative {total:>5}")

In [ ]:
import matplotlib.pyplot as plt

x = range(1, len(cumulative) + 1)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x, sent_per_turn, alpha=0.4, label="Tokens sent this turn")
ax.plot(x, cumulative, marker="o", color="crimson", label="Cumulative tokens")
ax.set_xlabel("Conversation turn")
ax.set_ylabel("Tokens")
ax.set_title("What it costs to 'remember'")
ax.set_xticks(list(x))
ax.legend()
ax.grid(alpha=0.3)
plt.show()

---
## 6. Gradio: the same loop, with a UI

`gr.ChatInterface` stores the history and hands it to us as `history`. The function below does exactly what we did by hand: **concatenate the list and resend it**.

In [ ]:
import gradio as gr

SYSTEM_PROMPT = "You are a helpful assistant. Keep answers short."


def chat(message, history):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, *history, {"role": "user", "content": message}]

    stream = client.chat.completions.create(model=MODEL, messages=messages, stream=True)

    partial = ""
    for chunk in stream:
        partial += chunk.choices[0].delta.content or ""
        yield partial


gr.ChatInterface(chat, type="messages", title="We are the ones storing the history").launch()

---
## 7. Tools: giving the model hands

The model does not know today's weather: its weights were frozen at training time. Let's let it *ask* for that data.

First, not every free model supports *function calling*. Let's ask the catalogue:

In [ ]:
import requests

catalogue = requests.get("https://openrouter.ai/api/v1/models", timeout=30).json()["data"]

tool_models = sorted(
    m["id"]
    for m in catalogue
    if m["id"].endswith(":free") and "tools" in m.get("supported_parameters", [])
)

print(len(tool_models), "free models with tool support\n")
for model_id in tool_models[:15]:
    print(" -", model_id)

TOOL_MODEL = tool_models[0] if tool_models else MODEL
print("\nWe will use:", TOOL_MODEL)

### The tool is an ordinary Python function

No AI here. We use [Open-Meteo](https://open-meteo.com/), which needs no API key.

In [ ]:
def get_current_weather(city: str) -> dict:
    """Return the current weather for a city using Open-Meteo."""
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1},
        timeout=10,
    ).json()

    if not geo.get("results"):
        return {"error": f"Unknown city: {city}"}

    place = geo["results"][0]
    weather = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": place["latitude"],
            "longitude": place["longitude"],
            "current": "temperature_2m,wind_speed_10m",
            "timezone": "auto",
        },
        timeout=10,
    ).json()["current"]

    return {
        "city": place["name"],
        "country": place.get("country"),
        "temperature_c": weather["temperature_2m"],
        "wind_kmh": weather["wind_speed_10m"],
    }


get_current_weather("Wellington")

### We describe it to the model

The schema is text that goes inside the prompt. The model only ever sees a description.

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name, e.g. Wellington"}
                },
                "required": ["city"],
            },
        },
    }
]

messages = [{"role": "user", "content": "What's the weather like in Auckland right now?"}]

first_pass = client.chat.completions.create(model=TOOL_MODEL, messages=messages, tools=TOOLS)
reply = first_pass.choices[0].message

print("Text       :", repr(reply.content))
print("Tool calls :", reply.tool_calls)

> ### 💡 The most misunderstood part
> **The LLM executed nothing.** It returned JSON *asking* us to call the function.
> The `requests.get` is done by our code, not by the model. An "AI app" is basically this loop.

In [ ]:
import json

AVAILABLE_TOOLS = {"get_current_weather": get_current_weather}

messages.append(reply.model_dump(exclude_none=True))

for call in reply.tool_calls or []:
    args = json.loads(call.function.arguments)
    result = AVAILABLE_TOOLS[call.function.name](**args)
    print(f"→ {call.function.name}({args}) = {result}")

    messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})

second_pass = client.chat.completions.create(model=TOOL_MODEL, messages=messages)
print("\n" + second_pass.choices[0].message.content)

### Plan B: the same idea without `tools`

If the free model does not support function calling, we can ask for the JSON in the prompt. It is more fragile, but it shows that **`tools` is syntactic sugar over "ask the model to reply in a given format"**.

In [ ]:
INSTRUCTIONS = (
    "You can use one function: get_current_weather(city).\n"
    'If the question needs it, reply ONLY with: {"tool": "get_current_weather", "city": "<city>"}\n'
    "Otherwise reply with plain text."
)
QUESTION = "What's the weather like in Queenstown?"

raw = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": INSTRUCTIONS},
        {"role": "user", "content": QUESTION},
    ],
    temperature=0,
).choices[0].message.content.strip()

print("The model replied:", raw)

try:
    request = json.loads(raw)
except json.JSONDecodeError:
    print("\nIt did not return JSON — this is exactly the fragility of the manual approach.")
else:
    observation = get_current_weather(request["city"])
    print("Tool result:", observation)

    final = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": QUESTION},
            {"role": "user", "content": f"Weather data: {json.dumps(observation)}. Answer in one sentence."},
        ],
    )
    print("\n" + final.choices[0].message.content)

---
## Recap

| What it looks like | What it really is |
|---|---|
| The chatbot remembers me | We resend the full list on every turn |
| The model is creative | It samples from a probability distribution |
| The model browses the internet | It returns JSON and *our* code makes the call |

It all comes down to **what text we put in the prompt**.

And that is the next problem: context is finite and you pay for it. We cannot paste a company's entire documentation into every question.

➡️ **Continues in [`02_rag.ipynb`](02_rag.ipynb)**